In [ ]:
import numpy as np
from astropy import units, constants
from scipy import integrate

# ─────────────────────────────────────────────────────────────────────────────
# Axion model parameters
# ─────────────────────────────────────────────────────────────────────────────

KSVZ = 1.92
DFSZ = 0.75

def g_x(C_ag, m_a):
    """Axion-photon coupling in GeV^-1, m_a in eV."""
    return 2e-10 * C_ag * m_a

# ─────────────────────────────────────────────────────────────────────────────
# Scan time
# ─────────────────────────────────────────────────────────────────────────────

s_per_year = 365.25 * 24 * 3600

def dB_to_eta(dB):
    """Convert dB backaction reduction to linear eta_A (amplitude convention)."""
    return 10**(dB / 20)

def scan_time(B0, V,
              model  = 'DFSZ',
              SNR    = 3,
              c_PU   = 0.1,
              Q      = 20e6,
              eta_A  = 0.1,
              T      = 10e-3,
              rho_DM = 0.45,
              g_ayy  = 1e-19,
              nu_min = 0.1e6,
              nu_max = 30e6):

    C_ag = DFSZ if model == 'DFSZ' else KSVZ

    common = (41e3 / s_per_year
              * (3      / SNR   )**2
              * (rho_DM / 0.45  )**2
              * (c_PU   / 0.1   )**4
              * (B0     / 16    )**4
              * (V      / 10    )**(10/3)
              * (Q      / 2e7   )
              * (10e-3  / T     )
              * (0.1    / eta_A ))

    def integrand(nu):
        if model is None:
            g = g_ayy
        else:
            ma_eV = constants.h.to('eV/Hz').value * nu
            g     = g_x(C_ag, ma_eV)
        dnu_dt = common * (g / 1e-19)**4 * (nu / 100e3)
        return 1.0 / dnu_dt

    result, _ = integrate.quad(integrand, nu_min, nu_max)
    return result / s_per_year

# ─────────────────────────────────────────────────────────────────────────────
# Comparison table
# ─────────────────────────────────────────────────────────────────────────────

print(f"{'Scenario':<40} {'Expected':>10}  {'DFSZ':>10} {'Ratio':>10}")
print("-" * 74)
scenarios = [
    (16, 10, 20e6, -20, 6.2, 'Baseline'),
    (29, 10, 20e6, -5, 3.2, 'Stronger magnet + higher noise'),
    (16, 8, 20e6, -25, 7.3, 'Lower noise + lower volume'),
    (16, 17, 2e6, -20, 10.6, 'Higher volume + lower Q'),
    (26, 10, 2e6, -20, 8.9, 'Stronger magnet + lower Q'),
]

for (B0, V, Q, eta_dB, expected, name) in scenarios:
    eta_A   = dB_to_eta(eta_dB)
    t_dfsz  = scan_time(B0, V, Q=Q, eta_A=eta_A, model='DFSZ')
    ratio  = expected/t_dfsz
    print(f"{name:<40} {expected:>10.1f} {t_dfsz:>10.2f} {ratio:>10.2f}")


Scenario                                   Expected        DFSZ      Ratio
--------------------------------------------------------------------------
Baseline                                        6.2       6.25       0.99
Stronger magnet + higher noise                  3.2       3.26       0.98
Lower noise + lower volume                      7.3       7.40       0.99
Higher volume + lower Q                        10.6      10.66       0.99
Stronger magnet + lower Q                       8.9       8.97       0.99
